## "Pre-process Once" strategy

In [1]:
import os

# Instead of -1, leave room for VS Code and Windows
total_cores = os.cpu_count()
safe_cores = max(1, total_cores - 2) 

print(f"Total CPU cores: {total_cores}")
print(f"Safe CPU cores: {safe_cores}")

Total CPU cores: 4
Safe CPU cores: 2


In [3]:
import pandas as pd
import numpy as np
import re
import spacy
import nltk
from tqdm import tqdm
from unicodedata import normalize
from nltk.corpus import stopwords
from spacy.attrs import LEMMA, IS_ALPHA
from sklearn.base import BaseEstimator, TransformerMixin

# Download necessary NLTK data
try:
    pt_stopwords = set(stopwords.words("portuguese"))
except LookupError:
    nltk.download("stopwords")
    pt_stopwords = set(stopwords.words("portuguese"))

# Load optimized spaCy model
spacy_nlp = spacy.load("pt_core_news_sm", disable=["parser", "ner", "textcat", "attribute_ruler"])

# Precompile Regex for speed
URL_RE = re.compile(r"http\S+")
MENTION_RE = re.compile(r"@\S+")
RT_RE = re.compile(r"RT @[\w_]+:")
NUM_RE = re.compile(r"\d+")
SPACE_RE = re.compile(r"\s+")

In [2]:
import os
from packaging import metadata
import pandas as pd

def load_fakebr(path):
    texts = []
    labels = []
    metadata = []

    meta_columns = [
    "author"
    ,"link"
    ,"category"
    ,"date of publication"
    ,"number of tokens"
    ,"number of words without punctuation"
    ,"number of types"
    ,"number of links inside the news"
    ,"number of words in upper case"
    ,"number of verbs"
    ,"number of subjuntive and imperative verbs"
    ,"number of nouns"
    ,"number of adjectives"
    ,"number of adverbs"
    ,"number of modal verbs (mainly auxiliary verbs)"
    ,"number of singular first and second personal pronouns"
    ,"number of plural first personal pronouns"
    ,"number of pronouns"
    ,"pausality"
    ,"number of characters"
    ,"average sentence length"
    ,"average word length"
    ,"percentage of news with speeling errors"
    ,"emotiveness"
    ,"diversity"]

    for label_type in ['fake', 'true']:
        folder_path = os.path.join(path, label_type)

        for filename in os.listdir(folder_path):
            file_path = os.path.join(folder_path, filename)

            with open(file_path, 'r', encoding='utf-8') as f:
                texts.append(f.read())
                labels.append(0 if label_type == 'fake' else 1)

    for label_type in ['fake', 'true']:
        folder_path = os.path.join(path, label_type +'-meta-information')

        for filename in os.listdir(folder_path):
            file_path = os.path.join(folder_path, filename)

            with open(file_path, 'r', encoding='utf-8') as f:
                lines = [l.strip() for l in f.readlines()]
                #print(lines)  
                data = dict(zip(meta_columns, lines))

                metadata.append(data)

    df = pd.DataFrame({
        'text': texts,
        'label': labels,
    })
    df_meta = pd.DataFrame(metadata)
    df = pd.concat([df, df_meta], axis=1)

    return df

# Caminho para os textos completos
data_path = "Fake.br-Corpus/full_texts"

df = load_fakebr(data_path)

print(df.shape)

df.head(1)

(7200, 27)


,text,label,author,link,category,date of publication,number of tokens,number of words without punctuation,number of types,number of links inside the news,...,number of singular first and second personal pronouns,number of plural first personal pronouns,number of pronouns,pausality,number of characters,average sentence length,average word length,percentage of news with speeling errors,emotiveness,diversity
0,EBC corrige matéria do Diário do Brasil. Marid...,0,None,https://www.diariodobrasil.org/paradeiro-de-ei...,politica,26/01/2017,141,122,89,2,...,0,0,5,2.375,587,15.25,4.81148,0.0,0.222222,0.729508


In [4]:
from sklearn.model_selection import train_test_split

# First split: 70% train, 30% temp
train_df, temp_df = train_test_split(
    df,
    test_size=0.30,
    stratify=df["label"],  # keeps fake/true balance
    random_state=42
)

# Second split: 10% val, 20% test
val_df, test_df = train_test_split(
    temp_df,
    test_size=2/3,  # because 20/(10+20) = 0.666...
    stratify=temp_df["label"],
    random_state=42
)

print(len(train_df), len(val_df), len(test_df))

5040 720 1440


In [5]:
%%time
import re
import spacy
import nltk
import numpy as np
from unicodedata import normalize
from nltk.corpus import stopwords
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.base import BaseEstimator, TransformerMixin
from spacy.attrs import LEMMA, IS_ALPHA
from tqdm import tqdm  # Recommended for progress tracking

# --------------------------------------------------
# 1. LOAD OPTIMIZED SPACY MODEL
# --------------------------------------------------
# Disable unnecessary components to save CPU cycles
spacy_nlp = spacy.load(
    "pt_core_news_sm", 
    disable=["parser", "ner", "textcat", "attribute_ruler"]
)
spacy_nlp.max_length = 3_000_000

# --------------------------------------------------
# 2. GLOBAL CONSTANTS & REGEX
# --------------------------------------------------
try:
    pt_stopwords = set(stopwords.words("portuguese"))
except LookupError:
    nltk.download("stopwords")
    pt_stopwords = set(stopwords.words("portuguese"))

URL_RE = re.compile(r"http\S+")
MENTION_RE = re.compile(r"@\S+")
RT_RE = re.compile(r"RT @[\w_]+:")
NUM_RE = re.compile(r"\d+")
SPACE_RE = re.compile(r"\s+")

# --------------------------------------------------
# 3. FAST CLEANING (Used inside the generator)
# --------------------------------------------------
def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = URL_RE.sub(" ", text)
    text = MENTION_RE.sub(" ", text)
    text = RT_RE.sub(" ", text)
    text = NUM_RE.sub(" ", text)
    # Only use normalize if you absolutely need to strip accents for TF-IDF
    text = normalize("NFKD", text).encode("ascii", "ignore").decode("ascii")
    return SPACE_RE.sub(" ", text).strip()

# --------------------------------------------------
# 4. HIGH-SPEED SPACY PREPROCESSOR (Integer-based)
# --------------------------------------------------
class SpacyPreprocessor(BaseEstimator, TransformerMixin):
    def __init__(self, max_tokens=100, batch_size=512, n_process=-1, show_progress=True):
        self.max_tokens = max_tokens
        self.batch_size = batch_size
        self.n_process = n_process
        self.show_progress = show_progress

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        # Localize for speed
        vocab = spacy_nlp.vocab
        stop_ids = {vocab.strings[w] for w in pt_stopwords if w in vocab.strings}

        
        # Create generator for parallel cleaning
        cleaned_gen = (clean_text(t) for t in X)
        
        # Setup progress bar
        total = len(X) if hasattr(X, '__len__') else None
        pipe_gen = spacy_nlp.pipe(
            cleaned_gen, 
            batch_size=self.batch_size, 
            n_process=self.n_process
        )
        
        if self.show_progress:
            pipe_gen = tqdm(pipe_gen, total=total, desc="Preprocessing Portuguese Text")

        processed = []
        with spacy_nlp.select_pipes(enable=["lemmatizer"]): #specifically enabling only what is strictly necessary for lemmatization.
            for doc in pipe_gen:
                # CORRECTED: Use integer attributes directly
                # This returns a 2D numpy array [n_tokens, 2]
                attr_array = doc.to_array([LEMMA, IS_ALPHA])
                
                if attr_array.size == 0:
                    processed.append("")
                    continue

                # Filtering using the IDs
                # row[0] is LEMMA ID, row[1] is IS_ALPHA (1 for True, 0 for False)
                lemmas = [
                    vocab.strings[row[0]] 
                    for row in attr_array 
                    if row[1] == 1 and row[0] not in stop_ids
                ]
                
                processed.append(" ".join(lemmas[:self.max_tokens]))
                # Periodically clear memory if processing millions of rows
                if len(processed) % 5000 == 0:
                    import gc
                    gc.collect()

            return processed

CPU times: user 297 ms, sys: 49 ms, total: 346 ms
Wall time: 339 ms


In [ ]:
%%time
import re
import spacy
import nltk
import pandas as pd
import numpy as np
from unicodedata import normalize
from nltk.corpus import stopwords
from sklearn.base import BaseEstimator, TransformerMixin
from spacy.attrs import LEMMA, IS_ALPHA
from tqdm import tqdm

# --------------------------------------------------
# 1. LOAD OPTIMIZED SPACY MODEL
# --------------------------------------------------
spacy_nlp = spacy.load(
    "pt_core_news_sm", 
    disable=["parser", "ner", "textcat", "attribute_ruler"]
)
spacy_nlp.max_length = 3_000_000

# --------------------------------------------------
# 2. GLOBAL CONSTANTS & REGEX
# --------------------------------------------------
try:
    pt_stopwords = set(stopwords.words("portuguese"))
except LookupError:
    nltk.download("stopwords")
    pt_stopwords = set(stopwords.words("portuguese"))

URL_RE = re.compile(r"http\S+")
MENTION_RE = re.compile(r"@\S+")
RT_RE = re.compile(r"RT @[\w_]+:")
NUM_RE = re.compile(r"\d+")
SPACE_RE = re.compile(r"\s+")

# --------------------------------------------------
# 3. FAST CLEANING FUNCTION
# --------------------------------------------------
def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = URL_RE.sub(" ", text)
    text = MENTION_RE.sub(" ", text)
    text = RT_RE.sub(" ", text)
    text = NUM_RE.sub(" ", text)
    # Strip accents
    text = normalize("NFKD", text).encode("ascii", "ignore").decode("ascii")
    return SPACE_RE.sub(" ", text).strip()

# --------------------------------------------------
# 4. HIGH-SPEED SPACY PREPROCESSOR
# --------------------------------------------------
class SpacyPreprocessor(BaseEstimator, TransformerMixin):
    def __init__(self, max_tokens=100, batch_size=1024, n_process=-1, show_progress=True):
        self.max_tokens = max_tokens
        self.batch_size = batch_size
        self.n_process = n_process
        self.show_progress = show_progress

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        vocab = spacy_nlp.vocab
        stop_ids = {vocab.strings[w] for w in pt_stopwords if w in vocab.strings}
        
        # Generator for parallel cleaning
        cleaned_gen = (clean_text(t) for t in X)
        
        total = len(X) if hasattr(X, '__len__') else None
        pipe_gen = spacy_nlp.pipe(
            cleaned_gen, 
            batch_size=self.batch_size, 
            n_process=self.n_process
        )
        
        if self.show_progress:
            pipe_gen = tqdm(pipe_gen, total=total, desc="Lemmatizing Portuguese Text")

        processed = []
        # Optimization: specifically enabling only what is strictly necessary
        with spacy_nlp.select_pipes(enable=["lemmatizer"]):
            for doc in pipe_gen:
                attr_array = doc.to_array([LEMMA, IS_ALPHA])
                
                if attr_array.size == 0:
                    processed.append("")
                    continue

                # Filtering using integer IDs (row[0]=LEMMA, row[1]=IS_ALPHA)
                lemmas = [
                    vocab.strings[row[0]] 
                    for row in attr_array 
                    if row[1] == 1 and row[0] not in stop_ids
                ]
                
                processed.append(" ".join(lemmas[:self.max_tokens]))
                # Periodically clear memory if processing millions of rows
                if len(processed) % 5000 == 0:
                    import gc
                    gc.collect()

        return processed

# --------------------------------------------------
# 5. DATASET GENERATION EXECUTION
# --------------------------------------------------

# Assuming train_df is already loaded in your environment
# Example: train_df = pd.read_csv("data.csv")

# Initialize Preprocessor
preprocessor = SpacyPreprocessor(max_tokens=100, batch_size=1024, n_process=-1)

# --- GENERATE DATASET 1: LEMMATIZED ---
print("--- Starting Lemmatization (Deep Clean) ---")
train_df["text_lemma"] = preprocessor.transform(train_df["text"])

# --- GENERATE DATASET 2: NO LEMMATIZATION ---
print("\n--- Starting Simple Regex Cleaning ---")
# Direct apply is faster than spinning up a pipe for simple regex
train_df["text_simple"] = train_df["text"].progress_apply(clean_text) if 'tqdm' in globals() else train_df["text"].apply(clean_text)

# --------------------------------------------------
# 6. SAVE OUTPUTS
# --------------------------------------------------

# Saving as Parquet to preserve structure and speed up future loading
train_df[["text_lemma", "label"]].to_parquet("train_lemmatized.parquet", index=False)
train_df[["text_simple", "label"]].to_parquet("train_simple_cleaned.parquet", index=False)

print("\nSuccess! Two datasets generated:")
print("- train_lemmatized.parquet (Lemmas, No Stopwords, Alpha only)")
print("- train_simple_cleaned.parquet (Regex cleaning, No Lemmatization)")